In [3]:
# ============================================================
# SETUP: Install all required packages
# ============================================================
# sklearn-crfsuite: CRF implementation for sequence labeling
# seqeval: entity-level evaluation metrics (the correct way to evaluate NER)
# transformers: HuggingFace library for Transformer models
# datasets: HuggingFace datasets library for loading NER benchmarks

!pip install sklearn-crfsuite seqeval transformers datasets accelerate tensorflow

     ---------------------------------------- 0.0/43.6 kB ? eta -:--:--
     ---------------------------------------- 43.6/43.6 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/383.7 kB ? eta -:--:--
   ------- -------------------------------- 71.7/383.7 kB 2.0 MB/s eta 0:00:01
   ------------------ --------------------- 174.1/383.7 kB 2.6 MB/s eta 0:00:01
   ------------------- -------------------- 184.3/383.7 kB 1.6 MB/s eta 0:00:01
   ---------------------------------------  378.9/383.7 kB 2.0 MB/s eta 0:00:01
   ---------------------------------------- 383.7/383.7 kB 1.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/303.1 kB ? eta -:--:--
   --------------------------------------- 303.1/303.1 kB 18.3 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16186 sha256=3e38de0ab64f4b084b99d7c5dfc0b5f70aa005

In [5]:
import sklearn_crfsuite
from sklearn_crfsuite import metrics as crf_metrics
from sklearn.model_selection import train_test_split
from seqeval.metrics import classification_report as seq_report
from seqeval.metrics import f1_score as seq_f1
from seqeval.scheme import IOB2
from sklearn.metrics import classification_report as sklearn_report
import pandas as pd
import numpy as np
import json

print("All packages imported!")

All packages imported!


In [7]:
# ============================================================
# LOAD TRAINING DATA: CoNLL format
# ============================================================
# We'll use the WNUT17 dataset (Emerging entities from social media)
# This dataset has entities that are harder to detect: new companies,
# creative works, products, etc.

def read_conll(filename):
    """
    Read a CoNLL-formatted file into a list of sentences.

    Each sentence is a list of (token, label) tuples.
    Sentences are separated by blank lines in the file.

    Returns:
        List[List[Tuple[str, str]]]: list of sentences,
        where each sentence is a list of (token, tag) pairs
    """
    sentences = []
    with open(filename, encoding="utf-8") as f:
        sentence = []
        for line in f:
            line = line.strip()
            if line:
                # Each non-empty line has: token<tab>label
                parts = line.split("\t")
                if len(parts) >= 2:
                    sentence.append((parts[0], parts[1]))
            else:
                # Blank line = end of sentence
                if sentence:
                    sentences.append(sentence)
                    sentence = []
        # Don't forget the last sentence (if file doesn't end with blank line)
        if sentence:
            sentences.append(sentence)
    return sentences

# Download WNUT17 data
import urllib.request
import os

wnut_url = "https://raw.githubusercontent.com/leondz/emerging_entities_17/master/wnut17train.conll"
if not os.path.exists("wnut17train.conll"):
    urllib.request.urlretrieve(wnut_url, "wnut17train.conll")
    print("Downloaded WNUT17 training data")

sentences = read_conll("wnut17train.conll")
print(f"Loaded {len(sentences)} sentences")
print(f"\nExample sentence:")
for token, tag in sentences[0]:
    marker = "  ←" if tag != "O" else ""
    print(f"  {token:<20} {tag}{marker}")

Downloaded WNUT17 training data
Loaded 3394 sentences

Example sentence:
  @paulwalk            O
  It                   O
  's                   O
  the                  O
  view                 O
  from                 O
  where                O
  I                    O
  'm                   O
  living               O
  for                  O
  two                  O
  weeks                O
  .                    O
  Empire               B-location  ←
  State                I-location  ←
  Building             I-location  ←
  =                    O
  ESB                  B-location  ←
  .                    O
  Pretty               O
  bad                  O
  storm                O
  here                 O
  last                 O
  evening              O
  .                    O


In [ ]:
# ============================================================
# FEATURE ENGINEERING: Define what the CRF sees for each token
# ============================================================
# CRF is NOT a neural network — it cannot read raw text.
# You must manually define a "feature dictionary" for every token.
# Think of it as filling out a form for each word in the sentence.
#
# The CRF then learns weights for each feature during training.
# For example, it might learn:
#   "If word.istitle() is True AND next word is also title case,
#    this token is probably B-PERSON"
#
# We define three groups of features:
#   1. Current word features — "What does this word look like?"
#   2. Previous word features — "What came before?" (left context)
#   3. Next word features — "What comes after?" (right context)
#
# This gives the CRF a window of 3 tokens (prev, current, next)
# but only through these hand-picked features. It never sees raw text.
# This is the key limitation vs Bi-LSTM (learns its own features)
# and Transformers (sees full sentence with pretrained knowledge).

def word2features(sent, i):
    """
    Extract features for the i-th token in a sentence.

    Features capture:
    1. Properties of the current word (shape, suffix, case)
    2. Properties of the previous word (context from the left)
    3. Properties of the next word (context from the right)
    4. Position markers (beginning/end of sentence)

    Args:
        sent: list of (token, tag) tuples
        i: index of the current token

    Returns:
        dict: feature name → feature value
    """
    word = sent[i][0]

    # ==========================================================
    # GROUP 1: Current word features — "What does this word look like?"
    # ==========================================================
    # Each feature captures a different signal about the word.
    # The CRF learns which combinations predict which entity tags.

    features = {
        # bias: always 1.0 — a constant baseline feature.
        # Lets the CRF learn a default tendency for each tag
        # (e.g., most tokens are "O", so the bias weight for "O" will be high)
        'bias': 1.0,

        # word.lower(): the word itself, lowercased.
        # Lets the CRF memorize specific words seen in training.
        # Example: "google" → likely ORG, "january" → likely DATE
        'word.lower()': word.lower(),

        # word[-3:] and word[-2:]: suffix features (last 2-3 characters).
        # Suffixes reveal word type without memorizing the whole word.
        # Examples: "-tion" = probably not entity, "-berg" = maybe PERSON,
        #           "-Inc" = maybe ORG, "-day" = maybe DATE
        'word[-3:]': word[-3:],
        'word[-2:]': word[-2:],

        # word.isupper(): is the word ALL CAPS?
        # ALL CAPS words are often acronyms = likely ORG or PRODUCT
        # Examples: NASA, FBI, GDP, NYSE, EBITDA
        'word.isupper()': word.isupper(),

        # word.istitle(): is the word Title Case (first letter capitalized)?
        # Title case is the strongest single signal for named entities.
        # Examples: "Thomas", "Google", "California", "January"
        # Non-entities are rarely title case mid-sentence.
        'word.istitle()': word.istitle(),

        # word.isdigit(): is the word a pure number?
        # Pure numbers appear in DATE ("2024"), MONEY ("500"), PERCENT ("15")
        # Examples: "2024" → DATE, "500" in "$500" → MONEY
        'word.isdigit()': word.isdigit(),

        # word.isalpha(): does the word contain ONLY letters?
        # False for "U.S.", "3rd", "$500", "Manuel-Miranda"
        # Helps distinguish clean words from mixed alphanumeric tokens
        'word.isalpha()': word.isalpha(),

        # word.length: character count of the word.
        # Very short words (1-2 chars) are rarely entities.
        # Very long words might be organization names or compounds.
        'word.length': len(word),

        # word.has_hyphen: does the word contain a hyphen?
        # Hyphenated words are often multi-part names: "Manuel-Miranda",
        # "Hewlett-Packard", or compound modifiers
        'word.has_hyphen': '-' in word,

        # word.has_dollar: does the word contain a dollar sign?
        # Strong signal for MONEY entities: "$500", "$81.8"
        'word.has_dollar': '$' in word,

        # word.has_at: does the word contain an @ sign?
        # Social media handles in the WNUT17 dataset: "@elonmusk"
        'word.has_at': '@' in word,
    }

    # ==========================================================
    # GROUP 2: Previous word features — "What came before?"
    # ==========================================================
    # Left context helps resolve ambiguity.
    # Examples:
    #   "Dr." before "Chen"       → PERSON (title signals a name follows)
    #   "in" before "California"  → LOC (preposition signals a location follows)
    #   "at" before "Google"      → ORG (preposition signals an org follows)
    #   "$" before "500"          → MONEY (currency symbol signals amount)
    #
    # If this is the FIRST token (i=0), there's no previous word,
    # so we set BOS=True (Beginning Of Sentence) instead.
    # BOS helps because sentence-initial words are often entities
    # in news/social media text (e.g., "Apple announced...")

    if i > 0:
        word1 = sent[i-1][0]
        features.update({
            '-1:word.lower()': word1.lower(),       # what word came before
            '-1:word.istitle()': word1.istitle(),   # was it title case (another name?)
            '-1:word.isupper()': word1.isupper(),   # was it all caps (acronym?)
        })
    else:
        features['BOS'] = True  # Beginning Of Sentence — no left context

    # ==========================================================
    # GROUP 3: Next word features — "What comes after?"
    # ==========================================================
    # Right context also resolves ambiguity.
    # Examples:
    #   "Apple" followed by "announced" → ORG (verb = company doing something)
    #   "Apple" followed by "pie"       → NOT an entity (food context)
    #   "Tim" followed by "Cook"        → B-PERSON (another name follows)
    #   "January" followed by "2025"    → DATE (year follows month)
    #
    # If this is the LAST token, there's no next word,
    # so we set EOS=True (End Of Sentence) instead.

    if i < len(sent) - 1:
        word1 = sent[i+1][0]
        features.update({
            '+1:word.lower()': word1.lower(),       # what word comes next
            '+1:word.istitle()': word1.istitle(),   # is it title case (continuation?)
            '+1:word.isupper()': word1.isupper(),   # is it all caps
        })
    else:
        features['EOS'] = True  # End Of Sentence — no right context

    return features

# ==========================================================
# HELPER FUNCTIONS: Convert full sentences
# ==========================================================
# These apply word2features to every token in a sentence

def sent2features(sent):
    """Extract features for all tokens in a sentence."""
    return [word2features(sent, i) for i in range(len(sent))]

def sent2labels(sent):
    """Extract labels for all tokens in a sentence."""
    return [label for token, label in sent]

def sent2tokens(sent):
    """Extract tokens for all tokens in a sentence."""
    return [token for token, label in sent]

# ==========================================================
# EXAMPLE: See what features look like for a real token
# ==========================================================
# This shows the actual feature dict the CRF receives for "Thomas"
# (first token in the first sentence of WNUT17)

print("Features for 'Thomas' (first token):")
features = word2features(sentences[0], 0)
for name, value in list(features.items()):
    print(f"  {name:<25} = {value}")
print(f"\nTotal features: {len(features)}")

# ==========================================================
# NOTE: You could add MORE features to improve CRF performance:
#   - POS tags (noun, verb, etc.) from spaCy or NLTK
#   - Word embeddings (dense vector features)
#   - Character n-grams (all 3-char substrings)
#   - Gazetteers (is this word in a known list of cities/companies?)
#   - Wider context window (-2, +2 tokens)
#
# But more features = more engineering effort, which is exactly
# why the field moved to neural models (Bi-LSTM, Transformers)
# that learn their own features automatically.
# ==========================================================